# 🏆 Image-to-3D Unified Benchmark
### TripoSR vs. LGM vs. TRELLIS — First Same-Protocol Comparison

**Purpose:** Run all three 3D generation engines on the same 100 GSO objects,
compute Chamfer Distance, F-Score, PSNR, LPIPS, and latency with a unified
protocol. Results feed directly into the research paper.

**GPU requirements:**
- TripoSR: T4 (16 GB) ✅  
- LGM: T4 (16 GB) ✅  
- TRELLIS: P100 (16 GB) or A100 recommended ✅  

**Kaggle settings:** Accelerator → GPU T4 x2, Internet → ON

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 1. Install shared dependencies ────────────────────────────────────────
!pip install -q trimesh scipy open3d lpips rembg huggingface-hub pandas pyrender
!pip install -q 'Pillow>=10.0' 'numpy<2.0'

In [ ]:
# ── 2. Install TripoSR ────────────────────────────────────────────────────
!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git
# Verify
import tsr; print('TripoSR OK')

In [ ]:
# ── 3. Install LGM ────────────────────────────────────────────────────────
!pip install -q kiui diffusers accelerate transformers
!pip install -q git+https://github.com/3DTopia/LGM.git
import lgm; print('LGM OK')

In [ ]:
# ── 4. Install TRELLIS ────────────────────────────────────────────────────
# TRELLIS needs spconv — use the prebuilt wheel matching your CUDA version
import torch
cuda_ver = torch.version.cuda.replace('.','')[:3]   # e.g. '121'
!pip install -q spconv-cu{cuda_ver} -f https://download.openmmlab.com/mmcv/dist/cu{cuda_ver}/torch{torch.__version__[:5]}/index.html 2>/dev/null || \
    pip install -q spconv-cu121  # fallback
!pip install -q git+https://github.com/microsoft/TRELLIS.git
import trellis; print('TRELLIS OK')

In [ ]:
# ── 5. Clone benchmark repo ───────────────────────────────────────────────
import os, sys
if not os.path.exists('/kaggle/working/image-to-3d-benchmark'):
    # Option A: clone from GitHub (replace with your repo URL)
    !git clone https://github.com/YOUR_USERNAME/image-to-3d-benchmark.git /kaggle/working/image-to-3d-benchmark
    
    # Option B: upload the zip to Kaggle dataset and unzip here
    # !unzip /kaggle/input/image-to-3d-benchmark/image-to-3d-benchmark.zip -d /kaggle/working/

sys.path.insert(0, '/kaggle/working/image-to-3d-benchmark')
!pip install -q -e /kaggle/working/image-to-3d-benchmark

In [ ]:
# ── 6. Download GSO dataset (100-object TripoSR curated subset) ───────────
# TripoSR provides a curated 100-object GSO eval split
!pip install -q gdown
import os
GSO_DIR = '/kaggle/working/data/gso'
if not os.path.exists(GSO_DIR):
    os.makedirs(GSO_DIR, exist_ok=True)
    # Download curated GSO subset (hosted by TripoSR team)
    # Replace the gdrive ID with the actual one from TripoSR README
    !gdown --fuzzy 'https://drive.google.com/drive/folders/YOUR_GSO_FOLDER_ID' \
        -O {GSO_DIR} --folder
    
    # Alternative: use huggingface-hub
    # from huggingface_hub import snapshot_download
    # snapshot_download('YOUR_HF_GSO_DATASET', local_dir=GSO_DIR)
    
print(f'GSO objects found: {len(os.listdir(GSO_DIR))}')

In [ ]:
# ── 7. Run TripoSR benchmark ──────────────────────────────────────────────
import torch
from src.engines.triposr_engine import TriposrEngine
from src.evaluation.benchmark_runner import BenchmarkRunner

torch.cuda.empty_cache()

triposr_engine = TriposrEngine(device='cuda')
triposr_engine.load_model()

runner_triposr = BenchmarkRunner(
    dataset_path=GSO_DIR,
    engines=[triposr_engine],
    num_samples=100,
    output_dir='/kaggle/working/outputs/triposr',
)
runner_triposr.run()
runner_triposr.save_results('/kaggle/working/outputs/triposr_results.json')

# Free VRAM before next engine
del triposr_engine
torch.cuda.empty_cache()

print(runner_triposr.get_summary_df())

In [ ]:
# ── 8. Run LGM benchmark ─────────────────────────────────────────────────
from src.engines.lgm_engine import LGMEngine

torch.cuda.empty_cache()

lgm_engine = LGMEngine(device='cuda')
lgm_engine.load_model()

runner_lgm = BenchmarkRunner(
    dataset_path=GSO_DIR,
    engines=[lgm_engine],
    num_samples=100,
    output_dir='/kaggle/working/outputs/lgm',
)
runner_lgm.run()
runner_lgm.save_results('/kaggle/working/outputs/lgm_results.json')

del lgm_engine
torch.cuda.empty_cache()

print(runner_lgm.get_summary_df())

In [ ]:
# ── 9. Run TRELLIS benchmark ──────────────────────────────────────────────
# NOTE: Use P100 or A100 accelerator for TRELLIS — T4 may OOM
from src.engines.trellis_engine import TrellisEngine

torch.cuda.empty_cache()

trellis_engine = TrellisEngine(
    device='cuda',
    output_format='mesh',
    slat_steps=12,      # reduce to 8 if OOM
    sparse_steps=12,
)
trellis_engine.load_model()

runner_trellis = BenchmarkRunner(
    dataset_path=GSO_DIR,
    engines=[trellis_engine],
    num_samples=100,
    output_dir='/kaggle/working/outputs/trellis',
)
runner_trellis.run()
runner_trellis.save_results('/kaggle/working/outputs/trellis_results.json')

del trellis_engine
torch.cuda.empty_cache()

print(runner_trellis.get_summary_df())

In [ ]:
# ── 10. Merge results & generate unified LaTeX table ─────────────────────
import json, pandas as pd

all_results = []
for path in [
    '/kaggle/working/outputs/triposr_results.json',
    '/kaggle/working/outputs/lgm_results.json',
    '/kaggle/working/outputs/trellis_results.json',
]:
    with open(path) as f:
        all_results.extend(json.load(f))

df = pd.DataFrame(all_results)
df.to_csv('/kaggle/working/outputs/unified_results.csv', index=False)

# Summary table
numeric_cols = ['latency_s','chamfer_distance','f_score_0.1','f_score_0.2','f_score_0.5','psnr','lpips']
existing = [c for c in numeric_cols if c in df.columns]
summary = df.groupby('engine')[existing].mean().round(4)
print('\n=== UNIFIED BENCHMARK RESULTS ===')
print(summary.to_string())

# LaTeX table
# Re-use benchmark runner's export method on the merged data
from src.evaluation.benchmark_runner import BenchmarkRunner
dummy = BenchmarkRunner.__new__(BenchmarkRunner)
dummy.results = all_results
latex = dummy.export_latex_table()
print('\n=== LaTeX TABLE ===')
print(latex)

with open('/kaggle/working/outputs/unified_table.tex', 'w') as f:
    f.write(latex)
print('\nSaved to /kaggle/working/outputs/unified_table.tex')

In [ ]:
# ── 11. Visualise Pareto frontier ─────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = {'TripoSR': '#2196F3', 'LGM': '#FF9800', 'TRELLIS': '#4CAF50'}

for engine_name, grp in df.groupby('engine'):
    c = colors.get(engine_name, 'grey')
    axes[0].scatter(
        grp['latency_s'].mean(), grp['f_score_0.1'].mean(),
        s=200, c=c, label=engine_name, zorder=5
    )
    axes[1].scatter(
        grp['latency_s'].mean(), grp['chamfer_distance'].mean(),
        s=200, c=c, label=engine_name, zorder=5
    )

axes[0].set_xlabel('Latency (s) →')
axes[0].set_ylabel('F-Score @ 0.1 ↑')
axes[0].set_title('Pareto: Speed vs. F-Score')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Latency (s) →')
axes[1].set_ylabel('Chamfer Distance ↓')
axes[1].set_title('Pareto: Speed vs. Chamfer Distance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/pareto_frontier.png', dpi=150)
plt.show()
print('Saved pareto_frontier.png')